# 06 — Generator Selection

Select one fine-tuned and one from-scratch generator under the approved post-benchmark amendment (Option B). This notebook reads results only; it loads no feature encoder, regenerates no images, and never accesses the test split.

## Load benchmark results, registry and active amendment

In [ ]:
from pathlib import Path
import json
import sys
import csv
ROOT = next(path for path in [Path.cwd(), *Path.cwd().parents] if (path / 'configs').is_dir())
sys.path.insert(0, str(ROOT))
sys.path.insert(0, str(ROOT / 'notebooks/utility'))
from notebooks.utility.generator_benchmark import load_protocol, load_registry, rank_generator_family
from notebooks.utility import gate_audit as ga
protocol = load_protocol(ROOT)
registry = load_registry(ROOT)
amendment = ga.load_active_amendment(ROOT)
confirmed_by_gen = ga.confirmed_duplicate_rates(ROOT)
metrics_path = ROOT / protocol['outputs']['metrics']
benchmark_rows = list(csv.DictReader(metrics_path.open())) if metrics_path.is_file() else []
paired_path = ROOT / protocol['outputs']['paired_differences']
if not paired_path.is_file():
    paired_path = ROOT / protocol['outputs']['root'] / 'gate_audit/paired_generator_differences.csv'
paired_rows = list(csv.DictReader(paired_path.open())) if paired_path.is_file() else []
{'active_amendment': protocol.get('active_amendment'),
 'amendment_status': amendment['status'] if amendment else None,
 'selected_policy': amendment['selected_policy'] if amendment else None,
 'post_benchmark': amendment['status'] == 'approved_post_benchmark' if amendment else None,
 'n_benchmark_rows': len(benchmark_rows), 'confirmed_duplicate_rates': confirmed_by_gen} if benchmark_rows else 'Not yet evaluated'


## Original gates outcome vs amended Option B safety gates

The original outcome (zero eligible under the preregistered coverage/pHash gates) is shown alongside the amended safety-gate eligibility. Coverage and pHash-only similarity are descriptive under the amendment, not binary gates.

In [ ]:
filtered_rows = [row for row in benchmark_rows if row.get('condition') == 'FILTERED']
original_finetuned = rank_generator_family(filtered_rows, 'finetuned', protocol['eligibility_gates']) if filtered_rows else []
original_fromscratch = rank_generator_family(filtered_rows, 'from_scratch', protocol['eligibility_gates']) if filtered_rows else []
amended_finetuned = ga.amended_family_ranking(filtered_rows, 'finetuned', amendment, confirmed_by_gen) if (filtered_rows and amendment) else []
amended_fromscratch = ga.amended_family_ranking(filtered_rows, 'from_scratch', amendment, confirmed_by_gen) if (filtered_rows and amendment) else []
original_outcome = {'eligible_under_original_gates': sum(bool(row['eligible']) for row in original_finetuned + original_fromscratch),
                    'exclusions': [(row['generator_id'], row['exclusion_reasons']) for row in original_finetuned + original_fromscratch]}
amended_outcome = {'eligible_under_amended_safety_gates': sum(bool(row['eligible']) for row in amended_finetuned + amended_fromscratch),
                   'amended_exclusions': [(row['generator_id'], row['amended_exclusion_reasons']) for row in amended_finetuned + amended_fromscratch],
                   'finetuned_rank': [(row['generator_id'], row['family_rank']) for row in amended_finetuned],
                   'from_scratch_rank': [(row['generator_id'], row['family_rank']) for row in amended_fromscratch]}
{'original': original_outcome, 'amended': amended_outcome}


## Descriptive metrics and KID-primary ranking

In [ ]:
display_columns = ['generator_id', 'family_rank', 'raddino_kid', 'raddino_kid_stability_low', 'raddino_kid_stability_high',
                   'raddino_coverage', 'raddino_precision', 'raddino_fid', 'inception_kid', 'raddino_kid_std',
                   'confirmed_duplicate_rate', 'perceptual_hash_duplicate_rate',  # descriptive only, not gates
                   'train_memorization_rate', 'synthetic_exact_duplicate_rate', 'provenance_manifest_valid',
                   'lineage_complete', 'training_corpus_manifest', 'generation_seconds_per_image', 'efficiency_status']
[[{column: row.get(column) for column in display_columns} for row in ranking] for ranking in (amended_finetuned, amended_fromscratch)]


## Paired differences and manual selection under the amendment

In [ ]:
SELECTED_FINETUNED_GENERATOR = "02_sd21_filtered_100steps"
SELECTED_FROM_SCRATCH_GENERATOR = "07_ldm_sdvae_extra1361"
PROPOSED_FINETUNED_GENERATOR = next((row['generator_id'] for row in amended_finetuned if row['eligible']), None)
PROPOSED_FROM_SCRATCH_GENERATOR = next((row['generator_id'] for row in amended_fromscratch if row['eligible']), None)
SELECTION_NOTES = ('Post-benchmark amendment v1 (Option B), human-approved: coverage-point and pHash-only removed as binary gates; '
                   'safety gates retained (exact/confirmed duplicate, train memorization, corruption, FILTERED validity, provenance, '
                   'lineage, test access). All five official candidates pass the safety gates; the preregistered KID-primary hierarchy '
                   'selects G02 (fine-tuned) and G07 (from-scratch). Downstream results must be interpreted with this amendment stated.')
{'selected': (SELECTED_FINETUNED_GENERATOR, SELECTED_FROM_SCRATCH_GENERATOR),
 'amended_proposed_top_rank': (PROPOSED_FINETUNED_GENERATOR, PROPOSED_FROM_SCRATCH_GENERATOR),
 'paired_generator_differences': paired_rows}


## Validate and save the selection (post-benchmark amendment)

In [ ]:
SAVE_SELECTION = True
if SAVE_SELECTION and benchmark_rows and amendment:
    output = ga.save_amended_selection(ROOT, SELECTED_FINETUNED_GENERATOR, SELECTED_FROM_SCRATCH_GENERATOR,
                                       benchmark_rows, notes=SELECTION_NOTES)
    print('Saved selection to', output)
    print(json.dumps(json.loads(Path(output).read_text()), indent=1))
else:
    print('Selection not saved. Requires benchmark results and an active amendment.')
